# 📊 Recommandations et restitution

> ✍️ **Notebook dirigé** · complétez les cellules marquées ✍️ À VOUS.

**Jour 2 · 13h30** · Préparation de la restitution évaluée

*Analyse exploratoire et interprétation métier*

**Objectifs**
- Formuler des recommandations opérationnelles structurées
- Intégrer les indicateurs ESG, les limites et les retours utilisateurs
- Préparer 15 minutes de restitution adaptées à chaque interlocuteur

In [1]:
# ⚙️ À exécuter en premier : récupère la configuration du cours (quelques secondes)
import os, sys
CONFIG_ID = "1bW5RsToUHc-7pGiyHS_1B2OYkAFsrv3r"

# Suppression du fichier corrompu s'il existe
if os.path.exists("config_projet.py"):
    os.remove("config_projet.py")

# Téléchargement propre avec gdown pour contourner l'avertissement de virus Google Drive
!gdown --id {CONFIG_ID} -O config_projet.py

print("✅ Configuration prête : chaque fichier de données se télécharge au premier usage")

✅ Configuration prête : chaque fichier de données se télécharge au premier usage


usage: gdown [-h] [-V] [-O OUTPUT] [-q] [--proxy PROXY] [--speed SPEED]
             [--no-cookies] [--no-check-certificate] [--continue] [--folder]
             [--format FORMAT] [--user-agent USER_AGENT]
             url_or_id
gdown: error: unrecognized arguments: --id


In [2]:
# Choisissez votre secteur : "commerce", "banque" ou "hotellerie"
SECTEUR = "banque"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config_projet import charger, reference, nettoyage_minimal

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
brut, cfg = charger(SECTEUR)
print(cfg["nom"], "·", brut.shape[0], "lignes,", brut.shape[1], "colonnes")

Crédit Azur · 20477 lignes, 20 colonnes


## 1. Les chiffres clés de votre étude
Rassemblez dans un tableau les cinq chiffres que vous citerez à l'oral.

In [3]:
df = nettoyage_minimal(brut, cfg).copy()
df["date_demande"] = pd.to_datetime(df["date_demande"])

date_v2 = pd.Timestamp("2024-03-01")
jeunes = df["tranche_age"].eq("18-24")

taux_avant = df.loc[
    jeunes & (df["date_demande"] < date_v2), "credit_accorde"
].mean()

taux_apres = df.loc[
    jeunes & (df["date_demande"] >= date_v2), "credit_accorde"
].mean()

part_auto_travaux = (
    df.loc[
        df["type_credit"].isin(["auto", "travaux"]),
        "pnb_estime_eur"
    ].sum()
    / df["pnb_estime_eur"].sum()
)

chiffres = pd.DataFrame({
    "Indicateur": [
        "Demandes après nettoyage",
        "Taux d'accord global",
        "Part auto + travaux dans le PNB estimé",
        "Taux d'accord des 18–24 ans avant V2",
        "Taux d'accord des 18–24 ans après V2"
    ],
    "Valeur": [
        f"{len(df):,}".replace(",", " "),
        f"{df['credit_accorde'].mean():.1%}",
        f"{part_auto_travaux:.1%}",
        f"{taux_avant:.1%}",
        f"{taux_apres:.1%}"
    ]
})

display(chiffres)

,Indicateur,Valeur
0,Demandes après nettoyage,20 384
1,Taux d'accord global,64.0%
2,Part auto + travaux dans le PNB estimé,83.9%
3,Taux d'accord des 18–24 ans avant V2,36.1%
4,Taux d'accord des 18–24 ans après V2,7.7%


## 2. Vos recommandations
Trois à cinq recommandations, chacune sur le même modèle.

| Constat chiffré | Recommandation | Impact attendu | Indicateur de suivi | Limite ou condition |
|---|---|---|---|---|
| Le taux d’accord des 18–24 ans passe de **36,1 % à 7,7 %** après V2. | L’équipe data et les risques doivent analyser sous **un mois** le poids du contrat et de l’ancienneté professionnelle dans les décisions. | Mieux comprendre la baisse des accords chez les jeunes et identifier les ajustements à tester. | Taux d’accord par âge, en comparant des profils financiers similaires, avec les effectifs. | L’écart observé ne prouve pas une discrimination. |
| Les crédits auto et travaux représentent **83,9 % du PNB estimé**. | La direction métier doit mettre en place sous **un mois** un suivi mensuel par type de crédit et par canal. | Identifier les produits et les canaux concernés par la baisse et mieux cibler les actions. | Nombre de demandes, taux d’accord, montants accordés et PNB estimé. | Le PNB estimé ne mesure pas le bénéfice net. Le taux d’incident doit également être suivi sur des crédits ayant une durée de suivi comparable. |
| Le taux d’accord est de **59,6 % pour les femmes**, contre **67,2 % pour les hommes**, soit **7,6 points d’écart**. | La conformité et les risques doivent organiser une première revue des écarts sous **trois mois**, puis chaque trimestre. | Repérer les écarts persistants et approfondir leurs causes. | Taux d’accord, écarts, ratios et effectifs par âge, genre et région. | Les différences de revenu, d’endettement et de situation professionnelle peuvent expliquer une partie des écarts. |

## 3. Les indicateurs ESG
Quels indicateurs environnementaux, sociaux ou de gouvernance suivre, au vu de votre analyse ?

Dans notre étude, les indicateurs ESG concernent surtout l’accès au crédit et le suivi des décisions de la banque.

| Dimension | Indicateur | Fréquence | Objectif |
|---|---|---|---|
| Sociale : accès au crédit | Taux d’accord par âge, genre et région, avec les effectifs | Chaque mois | Suivre les groupes qui obtiennent moins souvent un crédit. |
| Sociale : équité des décisions | Écarts et ratios entre les taux d’accord des groupes | Chaque trimestre | Vérifier si les différences diminuent ou augmentent, puis approfondir les comparaisons à profils financiers similaires. |
| Gouvernance : suivi du score | Taux d’accord et taux d’incident de remboursement | Chaque mois | Vérifier si la sélection plus stricte s’accompagne d’une baisse des incidents, sur des crédits ayant une durée de suivi comparable. |
| Gouvernance : traitement des alertes | Part des alertes analysées et documentées sous 30 jours | Chaque mois | Nous proposons une cible de 100 % des alertes examinées, avec une conclusion écrite. |

Les plaintes des jeunes actifs mentionnées dans le brief motivent notre analyse. Cependant, nous ne disposons pas des réclamations détaillées pour les étudier.

Les données disponibles ne permettent pas de construire un indicateur environnemental fiable.

**Conclusion :** ces indicateurs permettront de suivre l’accès au crédit, les écarts entre groupes et l’évolution du risque. Les écarts observés restent des signaux à analyser, et non des preuves de discrimination.

## 4. Adapter le discours


Nous adaptons la présentation des résultats aux questions de chaque interlocuteur.

| Interlocuteur | Ce qu’il veut savoir | Ce que nous lui montrons |
|---|---|---|
| Direction métier | Quels produits portent l’activité et quelles actions mettre en place ? | Les crédits auto et travaux représentent 83,9 % du PNB estimé. Nous proposons de suivre les demandes, les accords et les montants par produit et par canal pour mieux comprendre la baisse. |
| Décideurs | Quels sont les principaux résultats et quelle décision prendre ? | La baisse du PNB estimé et le recul du taux d’accord des 18–24 ans, de 36,1 % à 7,7 % après V2. Nous présentons ensuite les actions proposées, leurs échéances et leurs indicateurs. |
| Conformité | Quels groupes sont concernés et quelles sont les limites de l’analyse ? | Les taux d’accord par âge, genre et région, avec les effectifs. Nous expliquons que les écarts observés doivent être approfondis à profils financiers comparables et ne prouvent pas une discrimination. |

**Conclusion :** la direction métier a besoin d’actions concrètes, les décideurs d’une vue d’ensemble et la conformité d’une analyse des écarts et de leurs limites.

## 5. Le plan des 15 minutes

| Temps | Partie | Ce que nous présentons |
|---|---|---|
| 1 min | Le contexte | Crédit Azur a changé son score en mars 2024. Nous cherchons à comprendre la baisse d’activité et les différences d’accès au crédit. |
| 3 min | L’évolution de l’activité | La tendance du PNB estimé, les variations saisonnières et la rupture détectée autour de février–mars 2024. Cette proximité ne suffit pas à prouver que V2 explique seul la baisse. |
| 3 min | Les produits qui portent l’activité | Les crédits auto et travaux représentent 83,9 % du PNB estimé. Nous expliquons les différences entre les types de crédit. |
| 3 min | Les écarts entre les groupes | Le taux d’accord des 18–24 ans passe de 36,1 % à 7,7 % après V2. Nous présentons aussi les écarts selon le genre et les limites de ces comparaisons. |
| 4 min | Nos recommandations | Analyser les critères de V2, suivre ensemble l’activité et le risque, puis examiner régulièrement les écarts entre groupes. Nous précisons les responsables, les échéances et les indicateurs. |
| 1 min | La décision proposée | Nous demandons de valider une analyse du score V2 sous un mois et un suivi mensuel, avant de décider d’un éventuel ajustement. |

## Les limites à rappeler

- Les écarts entre groupes ne prouvent pas une discrimination : les profils financiers doivent aussi être comparés.
- Les remboursements sont connus uniquement pour les crédits accordés. Le taux de vrais positifs de 100 % ne permet donc pas de conclure à une égalité des chances.
- Le PNB estimé ne correspond ni à un encaissement mensuel réel ni au bénéfice net.
- La population nationale de référence ne correspond pas forcément aux personnes susceptibles de demander un crédit.

Chaque membre du groupe prend la parole. Nous prévoyons ensuite 5 minutes de questions.

## ✅ Avant de passer
- [ ] Le dashboard HTML s'ouvre sur le poste de présentation.
- [ ] Chaque recommandation a un chiffre, un indicateur et une limite.
- [ ] Les biais sont présentés sans jargon : qui est concerné, de combien, pourquoi.
- [ ] La dernière version du notebook est enregistrée dans GitHub.